##  03 Find Project Root

## Goal

- Sometimes notebooks are executed from different folders.
- To build reliable paths, we can locate the **project root** automatically by searching upward for a marker such as `.git`.

## Idea

- If your project contains a `.git` folder, then that folder is a good marker for the project root.
- We can start at the current working directory and move upward until we find it.

In [1]:
from pathlib import Path

def find_project_root(marker=".git"):
    path = Path.cwd()

    while path != path.parent:
        if (path / marker).exists():
            return path
        path = path.parent

    raise RuntimeError(f"Project root not found. Could not locate marker: {marker}")

In [2]:
try:
    root = find_project_root()
    print("Project root found:")
    print(root)

    data_path = root / "data" / "countries.geojson"
    print("\nExample data path from root:")
    print(data_path)
    print("Exists:", data_path.exists())
except RuntimeError as e:
    print("Notice:")
    print(e)

Project root found:
/Users/josemarioomrqz/4543-5993-Spatial-Maps

Example data path from root:
/Users/josemarioomrqz/4543-5993-Spatial-Maps/data/countries.geojson
Exists: False


## Why This Matters

- A root-finding function makes notebooks more reliable because the path logic does not depend as heavily on where the notebook happened to start.

- This becomes especially useful once students begin using:

  - `src/` layouts
  - helper libraries
  - nested notebook folders
  - bigger projects

In [3]:
# Optional: inspect the current directory and its parents
current = Path.cwd()
print("Current directory and parents:")
for p in [current, *current.parents]:
    print(" -", p)

Current directory and parents:
 - /Users/josemarioomrqz/4543-5993-Spatial-Maps/Assignments_Completed/02-Missile_Geometry_202/_micro_lessons/00-Paths
 - /Users/josemarioomrqz/4543-5993-Spatial-Maps/Assignments_Completed/02-Missile_Geometry_202/_micro_lessons
 - /Users/josemarioomrqz/4543-5993-Spatial-Maps/Assignments_Completed/02-Missile_Geometry_202
 - /Users/josemarioomrqz/4543-5993-Spatial-Maps/Assignments_Completed
 - /Users/josemarioomrqz/4543-5993-Spatial-Maps
 - /Users/josemarioomrqz
 - /Users
 - /


## Exercise A

Answer without writing code:

1. What happens when `find_project_root()` is called in a project with no `.git` folder? **ANS:** If find_project_root() searches upward for a .git folder and never finds one, it will typically keep checking parent directories until it reaches the filesystem root, then fail.
   
   
2. Name two other files or folders that could serve as a reliable project root marker.**ANS:** pyproject.toml or setup.py


3. Why is locating the root this way more robust than assuming the notebook always runs from the same folder? **ANS:** This is more robust because the notebook may run from different working directories. Instead of a assuming a fixed launch folder, the function searches upward from Path.cwd() and finds the real project root based on a marker that belongs to the project structure. 

## Exercise B

Call `find_project_root()` with a different marker.

1. Try `"pyproject.toml"` — does it find a root? What does that tell you? **ANS:** pyproject.toml does find a root which shows that find_project_root() can use other reliable marker files besides .git.
   
2. Try a marker that definitely doesn't exist (e.g. `"banana"`) — what happens, and why is the `RuntimeError` message useful? **ANS:** Because the function searched upward and never finds that marker. The error message is useful because it tell you exactly which marker could not be located
   

In [5]:
from pathlib import Path

# Change the marker to "pyproject.toml" and test what happens.
try: 
    root = find_project_root("pyproject.toml")
    print("Root found:")
    print(root)

except RuntimeError as e:
    print("Error:")
    print(e)

print()
# Then try a marker that definitely doesn't exist (e.g. "banana").

try: 
    root = find_project_root("banana")
    print("Root found:")
    print(root)

except RuntimeError as e:
    print("Error:")
    print(e)


print()
# Your code here

Root found:
/Users/josemarioomrqz/4543-5993-Spatial-Maps/Assignments_Completed/02-Missile_Geometry_202

Error:
Project root not found. Could not locate marker: banana



## Exercise C

Use `find_project_root()` to build a reliable path to this module's `countries.geojson` data file.

Use the directory tree printed above to figure out the correct subfolders from root, then replace `"???"` in the cell below.

In [7]:
from pathlib import Path

root = find_project_root()

# Build the path from root to this module's data folder and check countries.geojson
# Hint: look at the directory tree printed above to figure out the right subfolders
data_file = root / "Assignments_Completed" / "countries.geojson"

print("Looking for:", data_file)
print("Exists:", data_file.exists())

Looking for: /Users/josemarioomrqz/4543-5993-Spatial-Maps/Assignments_Completed/countries.geojson
Exists: False


## Optional Advanced — Multiple Markers and Custom Start

Extend `find_project_root` to accept:

- `markers` — a list of marker names; stop at the first directory that contains **any** of them
- `start` — an optional starting path (defaults to `Path.cwd()` if not given)

This lets you test the function from any location and makes it more flexible for projects that don't use git.

In [9]:
from pathlib import Path

def find_project_root_multi(markers=(".git", "pyproject.toml", "setup.py"), start: Path = None):
    """Return the first ancestor directory that contains any of the marker files/folders."""
    # Your code here
    path = Path.cwd() if start is None else Path(start)
    
    while path != path.parent:
        for marker in markers:
            if (path / marker).exists():
                return path
        path = path.parent

    for marker in markers:
        if (path / marker).exists():
            return path
        
    pass
    raise RuntimeError(f"Project root not found. Cannot locate a marker: {markers}")

root = find_project_root_multi()
print("Root:", root)

Root: /Users/josemarioomrqz/4543-5993-Spatial-Maps/Assignments_Completed/02-Missile_Geometry_202


## Check Your Understanding

1. `find_project_root()` walks upward until `path == path.parent`. What does that condition mean — when does it become true? **path == path.parent means the path has reached the filesystem root, where going to the parent does not move up any farther. That is when the loop stops.**


2. You call `find_project_root()` from a notebook and get back `/project`. You then write `root / "data" / "cities.json"`. What is the full absolute path that produces? **The full path is /project/data/cities.json**


3. A teammate hardcodes `Path("/Users/them/project/data/cities.json")` instead of using `find_project_root()`. What breaks when you run their notebook? **The hardcoded path is tied to a teammate's computer and folder layout. When you run the notebook, the path probably does not exist on your machine, so the notebook will fail to find the file.

---
# Summary

## Students should now understand

- what the working directory is
- how relative and absolute paths differ
- how to build paths into other folders
- how to check whether files exist
- how to locate a project root marker

## Recommended next step

Move into:

`01_JSON_GeoJSON`

because now students are ready to locate files **before** opening and inspecting them.

That tiny detail saves a lot of chaos later. A shocking amount, really.